<a href="https://colab.research.google.com/github/taralamb-lgtm/DATA230-Group-Project-SK-TL-ST/blob/main/notebook/rapids_viz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install RAPIDS
!git clone https://github.com/rapidsai-community/rapidsai-csp-utils.git
%cd rapidsai-csp-utils/colab
!bash rapids-colab.sh

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 629, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 629 (delta 149), reused 86 (delta 85), pack-reused 434 (from 3)
Receiving objects: 100% (629/629), 207.69 KiB | 6.92 MiB/s, done.
Resolving deltas: 100% (323/323), done.
/content/rapidsai-csp-utils/colab
PLEASE READ FOR 21.06
********************************************************************************************************
Another release, another script change.  We had to revise the script, which now:
1. Does a more comprehensive install
2. Includes BlazingSQL
3. is far easier for everyone to understand and maintain

The script will require you to add these 5 cells to your notebook.  We have also created a new startup template: 
https://colab.research.google.com/drive/1TAAi_szMfWqRfHVfjGSqnGVLr_ztzUM9?usp=sharing

CHANGES T
CELL 1:
    # This get the RAPIDS-Colab install files and test check 

In [ ]:
import cudf
import cuml
from numba import cuda

print("cuDF version:", cudf.__version__)
print("GPU:", cuda.gpus)

cuDF version: 26.02.01
GPU: <Managed Device 0>


In [ ]:
import cudf

url = "https://raw.githubusercontent.com/taralamb-lgtm/DATA230-Group-Project-SK-TL-ST/refs/heads/main/data/processed/crimes_cleaned.csv"

df = cudf.read_csv(url)

df.head()

,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,y_coordinate,year,updated_on,latitude,longitude,location,hour,day_of_week,month,is_weekend
0,14122826,JK164186,2026-02-24 00:00:00,056XX S KEELER AVE,1154,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT $300 AND UNDER,RESIDENCE,False,False,...,1867056.0,2026,2026-03-03 15:45:04,41.791120,-87.728006,"(41.791120495, -87.728006096)",0,Tuesday,2,0
1,14121197,JK161249,2026-02-24 00:00:00,048XX N NORDICA AVE,810,THEFT,OVER $500,STREET,False,False,...,1931843.0,2026,2026-03-03 15:45:04,41.969281,-87.802514,"(41.96928083, -87.802514207)",0,Tuesday,2,0
2,14119815,JK160176,2026-02-24 00:00:00,043XX W HIRSCH ST,486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,True,True,...,1908956.0,2026,2026-03-03 15:45:04,41.906144,-87.735429,"(41.906144038, -87.735428779)",0,Tuesday,2,0
3,14122502,JK163778,2026-02-24 00:00:00,032XX W WARREN BLVD,1152,DECEPTIVE PRACTICE,ILLEGAL USE CASH CARD,APARTMENT,False,False,...,1900164.0,2026,2026-03-03 15:45:04,41.881865,-87.707063,"(41.881865472, -87.707063458)",0,Tuesday,2,0
4,14122369,JK162722,2026-02-24 00:00:00,109XX S MORGAN ST,810,THEFT,OVER $500,STREET,False,False,...,1832115.0,2026,2026-03-03 15:45:04,41.694778,-87.647324,"(41.694778129, -87.647324077)",0,Tuesday,2,0


In [ ]:
print(df.shape)
print(df.columns)

(62723, 26)
Index(['id', 'case_number', 'date', 'block', 'iucr', 'primary_type',
       'description', 'location_description', 'arrest', 'domestic', 'beat',
       'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate',
       'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude',
       'location', 'hour', 'day_of_week', 'month', 'is_weekend'],
      dtype='object')


In [ ]:
# imports and helper to convert small cudf results to pandas
import cudf
import cupy as cp
import plotly.express as px
import plotly.io as pio
from IPython.display import HTML, display

def to_pandas_safe(gdf):
    # convert cudf DataFrame to pandas (small frames only)
    try:
        return gdf.to_pandas()
    except Exception as e:
        # fallback if already pandas
        if hasattr(gdf, "to_pandas"):
            return gdf.to_pandas()
        return gdf

In [ ]:
# top 10 crime types
top_crimes_gdf = df['primary_type'].value_counts().head(10).reset_index()
top_crimes_gdf.columns = ['primary_type','count']
top_crimes = to_pandas_safe(top_crimes_gdf)

fig_top = px.bar(top_crimes, x='count', y='primary_type', orientation='h',
                 title='Top 10 Most Common Crime Types',
                 labels={'count':'Number of incidents','primary_type':'Crime Type'},
                 height=450)
fig_top.update_layout(yaxis={'categoryorder':'total ascending'})
fig_top.show()

In [ ]:
# crimes by district (handle if district is numeric or categorical)
district_counts_gdf = df['district'].value_counts().reset_index()
district_counts_gdf.columns = ['district','count']
district_counts = to_pandas_safe(district_counts_gdf.head(20))  # show top 20

fig_district = px.bar(district_counts, x='district', y='count',
                      title='Top Districts by Crime Count',
                      labels={'district':'District','count':'Number of incidents'},
                      height=450)
fig_district.update_xaxes(type='category')
fig_district.show()

In [ ]:
# crimes by hour
if 'hour' not in df.columns:
    # create hour if you only have date
    df['hour'] = df['date'].dt.hour

hourly_gdf = df.groupby('hour').size().reset_index(name='count').sort_values('hour')
hourly = to_pandas_safe(hourly_gdf)

fig_hour = px.line(hourly, x='hour', y='count', title='Crimes by Hour of Day',
                   labels={'hour':'Hour of day (0-23)','count':'Number of incidents'}, markers=True, height=420)
fig_hour.update_xaxes(dtick=1)
fig_hour.show()

In [ ]:
import pandas as pd

In [ ]:
# ensure day_of_week exists (standardize names)
if 'day_of_week' not in df.columns:
    df['day_of_week'] = df['date'].dt.day_name()

# order days
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_gdf = df['day_of_week'].value_counts().reset_index()
dow_gdf.columns = ['day_of_week','count']
dow = to_pandas_safe(dow_gdf)
dow['day_of_week'] = pd.Categorical(dow['day_of_week'], categories=day_order, ordered=True)
dow = dow.sort_values('day_of_week')

fig_dow = px.bar(dow, x='day_of_week', y='count', title='Crimes by Day of Week',
                 labels={'day_of_week':'Day','count':'Number of incidents'}, height=420)
fig_dow.show()

In [ ]:
# sample to keep browser responsive
if 'latitude' in df.columns and 'longitude' in df.columns:
    sample_frac = 0.01 if len(df) > 100000 else 0.05
    sample_gdf = df[['latitude','longitude','primary_type']].sample(frac=sample_frac, random_state=1)
    sample_pd = to_pandas_safe(sample_gdf)
    fig_map = px.scatter_mapbox(sample_pd, lat='latitude', lon='longitude',
                                color='primary_type', hover_data=['primary_type'], zoom=10,
                                title='Sampled Crime Locations (map)', height=600)
    fig_map.update_layout(mapbox_style='open-street-map')
    fig_map.show()
else:
    print("No latitude/longitude columns found in dataset.")

In [ ]:
# arrest column assumed boolean or 0/1
if 'arrest' in df.columns:
    arrest_rate_gdf = df.groupby('primary_type')['arrest'].mean().reset_index(name='arrest_rate')
    arrest_rate_gdf = arrest_rate_gdf.sort_values('arrest_rate', ascending=False).head(15)
    arrest_rate = to_pandas_safe(arrest_rate_gdf)
    fig_arrest = px.bar(arrest_rate, x='arrest_rate', y='primary_type', orientation='h',
                       title='Arrest Rate by Crime Type (top 15)', labels={'arrest_rate':'Arrest rate','primary_type':'Crime Type'}, height=450)
    fig_arrest.show()
else:
    print("No 'arrest' column found in the dataset.")

In [ ]:
# collect HTML fragments
fragments = []
fragments.append(pio.to_html(fig_top, full_html=False, include_plotlyjs='cdn'))
fragments.append(pio.to_html(fig_district, full_html=False, include_plotlyjs=False))
fragments.append(pio.to_html(fig_hour, full_html=False, include_plotlyjs=False))
fragments.append(pio.to_html(fig_dow, full_html=False, include_plotlyjs=False))
fragments.append(pio.to_html(fig_map, full_html=False, include_plotlyjs=False) if 'fig_map' in globals() else "<p>Map not available</p>")
fragments.append(pio.to_html(fig_arrest, full_html=False, include_plotlyjs=False) if 'fig_arrest' in globals() else "<p>Arrest rate chart not available</p>")

dashboard_html = f"""
<!doctype html>
<html>
<head>
  <meta charset="utf-8"/>
  <title>Crime EDA Dashboard</title>
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
  <style> body {{ font-family: Arial, sans-serif; margin:12px; }} .card {{ border:1px solid #ddd; padding:12px; border-radius:8px; margin-bottom:12px; }} </style>
</head>
<body>
  <h1>Crime EDA Dashboard</h1>
  <p>Generated with RAPIDS + Plotly (Colab)</p>
  <div class="card">{fragments[0]}</div>
  <div class="card">{fragments[1]}</div>
  <div class="card">{fragments[2]}</div>
  <div class="card">{fragments[3]}</div>
  <div class="card">{fragments[4]}</div>
  <div class="card">{fragments[5]}</div>
</body>
</html>
"""

out_path = "/content/crime_eda_dashboard.html"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(dashboard_html)

# Provide download link in Colab
from google.colab import files
files.download(out_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cudf
import pandas as pd
import json

# Load dataset from GitHub using RAPIDS
url = "https://raw.githubusercontent.com/taralamb-lgtm/DATA230-Group-Project-SK-TL-ST/main/data/processed/crimes_cleaned.csv"

df = cudf.read_csv(url)

# Ensure numeric types
for col in ["district","hour","latitude","longitude"]:
    df[col] = df[col].astype("float64")

# Clean data
core = df.dropna(subset=["district","hour","primary_type"])
core["district"] = core["district"].astype("int32")
core["hour"] = core["hour"].astype("int32")

# Convert to pandas for dashboard export
core_pd = core.to_pandas()

map_df = core_pd.dropna(subset=["latitude","longitude"])

# Sample map data for performance
if len(map_df) > 8000:
    map_df = map_df.sample(8000, random_state=42)

districts = sorted(core_pd["district"].unique())

records_core = core_pd[["district","hour","primary_type"]].to_dict("records")
records_map = map_df[["district","hour","latitude","longitude","primary_type"]].to_dict("records")

options_html = "".join(
    f'<option value="{d}">District {d}</option>' for d in districts
)

html = f"""
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>Interactive Crime Dashboard</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>

body {{
font-family: Arial;
margin: 25px;
background: #fafafa;
}}

h1 {{
margin-bottom: 15px;
}}

.controls {{
display:grid;
grid-template-columns:200px 1fr 1fr;
gap:15px;
background:white;
padding:15px;
border-radius:8px;
margin-bottom:20px;
}}

.card {{
background:white;
padding:10px;
border-radius:8px;
margin-bottom:20px;
}}

select,input {{
width:100%;
}}

</style>

</head>

<body>

<h1>Interactive Crime Dashboard</h1>

<div class="controls">

<div>
District
<select id="districtSelect">
<option value="All">All</option>
{options_html}
</select>
</div>

<div>
Start Hour
<input id="startHour" type="range" min="0" max="23" value="0">
</div>

<div>
End Hour
<input id="endHour" type="range" min="0" max="23" value="23">
</div>

</div>

<div class="card">
<div id="barChart"></div>
</div>

<div class="card">
<div id="lineChart"></div>
</div>

<div class="card">
<div id="mapChart"></div>
</div>

<script>

const coreData = {json.dumps(records_core)};
const mapData = {json.dumps(records_map)};

const districtSelect = document.getElementById("districtSelect");
const startHour = document.getElementById("startHour");
const endHour = document.getElementById("endHour");

function filterRows(rows) {{

const district = districtSelect.value;
const start = parseInt(startHour.value);
const end = parseInt(endHour.value);

return rows.filter(r => {{

const okDistrict = district==="All" || String(r.district)===district;
const okHour = r.hour >= start && r.hour <= end;

return okDistrict && okHour;

}});

}}

function buildBar(rows) {{

let counts = {{}};

rows.forEach(r=>{{
counts[r.primary_type]=(counts[r.primary_type]||0)+1;
}})

let entries = Object.entries(counts)
.sort((a,b)=>b[1]-a[1])
.slice(0,10)
.reverse();

Plotly.newPlot("barChart",[{{

type:"bar",
x:entries.map(d=>d[1]),
y:entries.map(d=>d[0]),
orientation:"h"

}}],{{

title:"Top 10 Crime Types",

margin:{{
l:220,
r:20,
t:60,
b:40
}},

yaxis:{{
automargin:true
}}

}})

}}

function buildLine(rows) {{

let hours = Array(24).fill(0)

rows.forEach(r=>{{
hours[r.hour]+=1
}})

Plotly.newPlot("lineChart",[{{

x:[...Array(24).keys()],
y:hours,
mode:"lines+markers"

}}],{{

title:"Crime by Hour",
margin:{{t:60}}

}})

}}

function buildMap(rows) {{

Plotly.newPlot("mapChart",[{{

type:"scattermapbox",
lat:rows.map(r=>r.latitude),
lon:rows.map(r=>r.longitude),
mode:"markers",
marker:{{size:7,opacity:0.6}}

}}],{{

title:"Crime Locations",
mapbox:{{style:"open-street-map",zoom:9,center:{{lat:41.88,lon:-87.63}}}}

}})

}}

function update() {{

const filteredCore = filterRows(coreData)
const filteredMap = filterRows(mapData)

buildBar(filteredCore)
buildLine(filteredCore)
buildMap(filteredMap)

}}

districtSelect.onchange = update
startHour.oninput = update
endHour.oninput = update

update()

</script>

</body>
</html>
"""

# Save dashboard
with open("interactive_crime_dashboard.html","w") as f:
    f.write(html)

print("Dashboard exported")

from google.colab import files
files.download("interactive_crime_dashboard.html")

Dashboard exported


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cudf
import pandas as pd
import json

# Load dataset from GitHub using RAPIDS
url = "https://raw.githubusercontent.com/taralamb-lgtm/DATA230-Group-Project-SK-TL-ST/main/data/processed/crimes_cleaned.csv"
df = cudf.read_csv(url)

# Make sure needed columns are numeric where appropriate
for col in ["district", "hour", "latitude", "longitude", "community_area"]:
    if col in df.columns:
        df[col] = cudf.to_numeric(df[col], errors="coerce")

# Keep rows needed for dashboard
needed = ["district", "hour", "primary_type", "latitude", "longitude"]
optional = [c for c in ["community_area"] if c in df.columns]
core_cols = needed + optional

core = df[core_cols].dropna(subset=["district", "hour", "primary_type"]).copy()
core["district"] = core["district"].astype("int32")
core["hour"] = core["hour"].astype("int32")

# Convert to pandas for HTML export / JS use
core_pd = core.to_pandas()

# Map data
map_df = core_pd.dropna(subset=["latitude", "longitude"]).copy()

# Sample map data so dashboard stays responsive
if len(map_df) > 8000:
    map_df = map_df.sample(8000, random_state=42)

# Keep top 6 crime types for a readable legend, group others as "Other"
top_types = core_pd["primary_type"].value_counts().head(6).index.tolist()
map_df["crime_group"] = map_df["primary_type"].apply(lambda x: x if x in top_types else "Other")

districts = sorted(core_pd["district"].dropna().unique().tolist())

records_core = core_pd[["district", "hour", "primary_type"]].to_dict("records")

map_cols = ["district", "hour", "latitude", "longitude", "primary_type", "crime_group"]
if "community_area" in map_df.columns:
    map_cols.append("community_area")
records_map = map_df[map_cols].to_dict("records")

options_html = "".join(
    f'<option value="{d}">District {d}</option>' for d in districts
)

html = f"""
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>Interactive Crime Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>
body {{
  font-family: Arial, sans-serif;
  margin: 25px;
  background: #fafafa;
}}
h1 {{
  margin-bottom: 8px;
}}
.subtitle {{
  color: #555;
  margin-bottom: 18px;
}}
.controls {{
  display: grid;
  grid-template-columns: 220px 1fr 1fr;
  gap: 15px;
  background: white;
  padding: 15px;
  border-radius: 8px;
  margin-bottom: 20px;
}}
.card {{
  background: white;
  padding: 12px;
  border-radius: 8px;
  margin-bottom: 20px;
}}
select, input {{
  width: 100%;
}}
.label {{
  font-weight: 600;
  margin-bottom: 6px;
}}
</style>
</head>

<body>

<h1>Interactive Crime Dashboard</h1>
<div class="subtitle">Filter by district and time of day to compare crime type, timing, and location patterns.</div>

<div class="controls">
  <div>
    <div class="label">District</div>
    <select id="districtSelect">
      <option value="All">All</option>
      {options_html}
    </select>
  </div>

  <div>
    <div class="label">Start Hour</div>
    <input id="startHour" type="range" min="0" max="23" value="0">
    <div><span id="startVal">0</span>:00</div>
  </div>

  <div>
    <div class="label">End Hour</div>
    <input id="endHour" type="range" min="0" max="23" value="23">
    <div><span id="endVal">23</span>:00</div>
  </div>
</div>

<div class="card">
  <div id="barChart"></div>
</div>

<div class="card">
  <div id="hourChart"></div>
</div>

<div class="card">
  <div id="mapChart"></div>
</div>

<script>
const coreData = {json.dumps(records_core)};
const mapData = {json.dumps(records_map)};

const districtSelect = document.getElementById("districtSelect");
const startHour = document.getElementById("startHour");
const endHour = document.getElementById("endHour");
const startVal = document.getElementById("startVal");
const endVal = document.getElementById("endVal");

function normalizeHours() {{
  let start = parseInt(startHour.value);
  let end = parseInt(endHour.value);

  if (start > end) {{
    const tmp = start;
    start = end;
    end = tmp;
    startHour.value = start;
    endHour.value = end;
  }}

  startVal.textContent = start;
  endVal.textContent = end;
  return [start, end];
}}

function filterRows(rows) {{
  const district = districtSelect.value;
  const [start, end] = normalizeHours();

  return rows.filter(r => {{
    const okDistrict = district === "All" || String(r.district) === district;
    const okHour = r.hour >= start && r.hour <= end;
    return okDistrict && okHour;
  }});
}}

function buildBar(rows) {{
  let counts = {{}};

  rows.forEach(r => {{
    counts[r.primary_type] = (counts[r.primary_type] || 0) + 1;
  }});

  let entries = Object.entries(counts)
    .sort((a,b) => b[1] - a[1])
    .slice(0, 10)
    .reverse();

  Plotly.newPlot("barChart", [{{
    type: "bar",
    x: entries.map(d => d[1]),
    y: entries.map(d => d[0]),
    orientation: "h",
    hovertemplate: "%{{y}}<br>Count: %{{x}}<extra></extra>"
  }}], {{
    title: "Top 10 Crime Types",
    margin: {{ l: 220, r: 20, t: 60, b: 40 }},
    yaxis: {{ automargin: true }},
    xaxis: {{ title: "Incident count" }}
  }}, {{
    responsive: true,
    displayModeBar: false
  }});
}}

function buildHour(rows) {{
  let hours = Array(24).fill(0);

  rows.forEach(r => {{
    if (r.hour >= 0 && r.hour <= 23) {{
      hours[r.hour] += 1;
    }}
  }});

  Plotly.newPlot("hourChart", [{{
    type: "bar",
    x: [...Array(24).keys()],
    y: hours,
    hovertemplate: "Hour %{{x}}:00<br>Count: %{{y}}<extra></extra>"
  }}], {{
    title: "Crime Count by Hour",
    margin: {{ l: 60, r: 20, t: 60, b: 50 }},
    xaxis: {{
      title: "Hour of day",
      tickmode: "array",
      tickvals: [...Array(24).keys()],
      ticktext: [...Array(24).keys()].map(h => h + ":00")
    }},
    yaxis: {{ title: "Incident count" }}
  }}, {{
    responsive: true,
    displayModeBar: false
  }});
}}

function buildMap(rows) {{
  const colorMap = {{
    "THEFT": "#1f77b4",
    "BATTERY": "#d62728",
    "CRIMINAL DAMAGE": "#2ca02c",
    "ASSAULT": "#9467bd",
    "MOTOR VEHICLE THEFT": "#ff7f0e",
    "DECEPTIVE PRACTICE": "#8c564b",
    "Other": "#7f7f7f"
  }};

  const groups = [...new Set(rows.map(r => r.crime_group))];
  const traces = groups.map(group => {{
    const subset = rows.filter(r => r.crime_group === group);
    return {{
      type: "scattermapbox",
      mode: "markers",
      name: group,
      lat: subset.map(r => r.latitude),
      lon: subset.map(r => r.longitude),
      marker: {{
        size: 7,
        opacity: 0.6,
        color: colorMap[group] || "#7f7f7f"
      }},
      text: subset.map(r => {{
        const ca = r.community_area !== undefined && r.community_area !== null
          ? "Community Area: " + r.community_area + "<br>"
          : "";
        return "Crime Type: " + r.primary_type +
               "<br>District: " + r.district +
               "<br>" + ca +
               "Hour: " + r.hour + ":00" +
               "<br>City: Chicago";
      }}),
      hovertemplate: "%{{text}}<extra></extra>"
    }};
  }});

  Plotly.newPlot("mapChart", traces, {{
    title: "Crime Locations by Type",
    margin: {{ l: 20, r: 20, t: 60, b: 10 }},
    mapbox: {{
      style: "open-street-map",
      zoom: 9,
      center: {{ lat: 41.88, lon: -87.63 }}
    }},
    legend: {{
      title: {{ text: "Crime Type" }},
      orientation: "h",
      y: -0.12
    }}
  }}, {{
    responsive: true,
    displayModeBar: false
  }});
}}

function update() {{
  const filteredCore = filterRows(coreData);
  const filteredMap = filterRows(mapData);

  buildBar(filteredCore);
  buildHour(filteredCore);
  buildMap(filteredMap);
}}

districtSelect.onchange = update;
startHour.oninput = update;
endHour.oninput = update;

update();
</script>

</body>
</html>
"""

with open("interactive_crime_dashboard.html", "w") as f:
    f.write(html)

print("Dashboard exported: interactive_crime_dashboard.html")

from google.colab import files
files.download("interactive_crime_dashboard.html")

Dashboard exported: interactive_crime_dashboard.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>